## Overview

This notebook is intended to:
* Introduce core views and functions available in Awesense's Energy Data Model (EDM).
* Demonstrate various ways to fetch and work with data using SQL and Python.

For complete documentation on the available views and functions, please refer to the [access_and_documentation.ipynb](../1_access_and_documentation/access_and_documentation.ipynb) notebook.

---

## Set up

In [1]:
import getpass
import urllib.parse

**Connection**

Enter the EDM server address and the login credentials provided by Awesense. If you do not have the credentials, or have any trouble connecting, please contact api@awesense.com.
<span style='color:red'> **Please do NOT store the credentials in the notebook, nor share them with anyone.** </span>

In [4]:
edm_address = getpass.getpass(prompt='EDM server address: ')

print('\nEDM login information')
edm_name = getpass.getpass(prompt='Username: ')
edm_password = getpass.getpass(prompt='Password: ')
edm_password = urllib.parse.quote(edm_password)

%load_ext sql
%sql postgresql://$edm_name:$edm_password@$edm_address/edm
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = False

# Delete the credential variables for security purpose.
del edm_name, edm_password

EDM server address: ········

EDM login information
Username: ········
Password: ········
The sql extension is already loaded. To reload it, use:
  %reload_ext sql


---

## Views

**Grid**

The term `grid` represents a portion of a utility's grid, and the term `grid element` represents the physical elements that comprise the grid.

The view named `grid` lists available grids in the database, and `grid_id` is the primary key of the grid.

In [5]:
%%sql

SELECT *
FROM grid;

grid_id,description,last_updated
awefice,awefice,2021-01-01 20:00:00+00:00
North Central Zone,North Central Zone,2021-01-01 20:00:00+00:00


**Grid Element**

The `grid_element` view contains information about grid elements in a grid. Properties that are common to all grid elements, such as `phases`, are explicitly listed as columns of the view. And additional metadata that may differ between grid elements of different types (e.g. meter vs. transformers, etc.) are stored as JSONB in the `meta` column. 

The `grid_id` column from the `grid_element` view references the `grid_id` from the `grid` view. And the `grid_id` and `grid_element_id` columns are often used as input arguments to various functions, as demonstrated in the **Functions** section below and in the [grid_tracing.ipynb](../3_grid_tracing/grid_tracing.ipynb) notebook. 

In [6]:
%%sql

SELECT *
FROM grid_element
WHERE grid_id = 'awefice'
LIMIT 1;

grid_id,grid_element_id,type,customer_type,phases,is_underground,is_producer,is_consumer,is_switchable,switch_is_open,terminal1_cn,terminal2_cn,power_flow_direction,upstream_grid_element_id,geometry,meta
awefice,batt_10,Battery,battery,ABC,None,True,True,False,None,con_135_t2,VCN.batt_10,both,None,0101000020E61000000B45A2539AC65EC0DDB431982CA24840,"{'name': 'batt_10', 'label': 'batt_10', 'sdp_id': None, 'lv_feeder': None, 'mv_feeder': None, 'ownership': 'private', 'make_model': 'LG CHEM RESU10H', 'enclosure_id': None, 'voltage_level': 'LV', 'capacity_usable': 9.6, 'commission_date': None, 'capacity_nominal': 9.8, 'manufacture_date': None, 'efficiency_round_trip': 97.96, 'max_peak_charge_power': 7.0, 'parent_transformer_id': 'transformer_36', 'max_peak_discharge_power': 7.0, 'max_continuous_charge_power': 5.0, 'max_continuous_discharge_power': 5.0}"


Below are different grid element types available in the `awefice` grid. 

A CSV file detailing all grid element types in the `Sandbox` server and their respective properties that the Awesense Platform recognizes can be found [here](https://sandbox.awesense.com/docs/_static/gis_metadata.csv).

In [7]:
%%sql

SELECT DISTINCT type as element_type
FROM grid_element
WHERE grid_id = 'awefice'
ORDER BY element_type;

element_type
ACLineSegment
Battery
Busbar
Cabinet
CircuitBreaker
Disconnector
EVCharger
Fuse
Jumper
Meter


A complete list of properties available in the `meta` column for a grid element can be returned using the built-in `json_each_text()` function. The following is an example of accessing each item in the `meta` column with the 'awefice' `grid_id` and the 'm_12' `grid_element_id`.

In [8]:
%%sql

SELECT ge.grid_element_id, 
        meta_data.key, 
        meta_data.value
FROM grid_element ge
JOIN json_each_text(ge.meta::json) AS meta_data
    ON true
WHERE grid_id = 'awefice'
    AND grid_element_id = 'm_14';

grid_element_id,key,value
m_14,name,None
m_14,label,m_14
m_14,sdp_id,None
m_14,address,"John Wilson, 56141 Christopher Vista Suite 631New Donnaton, VT 65768"
m_14,interval,3600
m_14,estimated,None
m_14,lv_feeder,None
m_14,mv_feeder,None
m_14,read_type,interval
m_14,tariff_id,res_basic


Specific properties from the `meta` column can be fetched individually as well.

In [9]:
%%sql

SELECT meta ->> 'address' as address,
       meta ->> 'meter_number' as mater_number
FROM grid_element 
WHERE grid_id = 'awefice'
    AND grid_element_id = 'm_12';

address,mater_number
"Alexander Caldwell, 36680 Nicole Pine Apt. 175New Susanville, MN 38456",D2-8809154B


The `geometry` field can be easily converted to `latitude` and `longitude` values using postGIS functions.

For grid elements that are single points, such as `meters`:

In [10]:
%%sql

SELECT ST_Y(geometry) as latitude,
        ST_X(geometry) as longitude      
FROM grid_element 
WHERE grid_id = 'awefice'
    AND grid_element_id = 'm_12';

latitude,longitude
49.2667165019629,-123.10284098634


For grid elements such as `line_segment` with starting and ending points, `geometry` can be used to display the `latitude` and `longitude` of the central point of the line as well as the lines' start and end coordinates:

In [11]:
%%sql

SELECT ST_Y(ST_Centroid(geometry)) as center_point_latitude,
    ST_X(ST_Centroid(geometry)) as center_point_longitude,
    ST_Y(ST_StartPoint(geometry)) as start_point_latitude, 
    ST_X(ST_StartPoint(geometry)) as start_point_longitude,
    ST_Y(ST_EndPoint(geometry)) as end_point_latitude, 
    ST_X(ST_EndPoint(geometry)) as end_point_longitude   
FROM grid_element
WHERE grid_id = 'awefice'
    AND grid_element_id = 'line_segment_81';

center_point_latitude,center_point_longitude,start_point_latitude,start_point_longitude,end_point_latitude,end_point_longitude
49.2678801500481,-123.105791066307,49.2678715685268,-123.105327920392,49.2679227270403,-123.106219178449


**Grid Element Data Source**

The `grid_element_data_source` view represents the linking between time-series data and physical elements on the grid. The primary key of this view is `grid_element_data_source_id`, and is used as an input argument of the `ts_data_source_select()` function to retrieve time series data, as demonstrated in the **Functions** section below and in the [time_series.ipynb](../4_time_series/time_series.ipynb) notebook.

In [12]:
%%sql

SELECT *
FROM grid_element_data_source
LIMIT 5;

grid_element_data_source_id,grid_id,grid_element_id,valid,phases,type,provider,metrics,friendly_id,direction
7a0f216a-ea10-4e47-afa2-aa9a486120b1,awefice,transformer_6,"[2022-01-01 08:00:00+00:00, 2023-12-08 08:00:00+00:00)",ABC,RISK,csv_file,"['unemployment_rate_7', 'energy_theft_count_7']",segment_factor_aggregates_7,None
883ac249-3b18-4772-854d-d31ec6659c11,awefice,transformer_2,"[2022-01-01 08:00:00+00:00, 2023-12-08 08:00:00+00:00)",ABC,RISK,csv_file,"['unemployment_rate_7', 'energy_theft_count_7']",segment_factor_aggregates_7,None
9eadb399-7cf0-43f6-a6a9-c9758edb2b69,awefice,switchable_element_4,"[2021-01-01 08:01:00+00:00, None)",ABC,SENSOR,awe_scada_reads,['kWh'],scada_sensor_4,None
6597c2d6-6d5b-44bd-b1f2-fc6abad12af3,awefice,transformer_16,"[2021-01-01 08:00:00+00:00, 2022-12-31 08:00:00+00:00)",A,RISK,csv_file,['number_of_outages'],number_of_outages,None
11f970dd-6a5c-498a-83b3-c308e085ecd0,awefice,transformer_2,"[2021-01-01 08:00:00+00:00, 2022-12-31 08:00:00+00:00)",ABC,RISK,csv_file,['number_of_outages'],number_of_outages,None


The `type` field refers to the class of stored data, and there may be multiple `type` values for a given grid element. To retrieve the exact time series data of interest, filter the `grid_element_data_source` by the `type` as below.

In [13]:
%%sql

SELECT *
FROM grid_element_data_source
WHERE type = 'CONSUMER'
LIMIT 5;

grid_element_data_source_id,grid_id,grid_element_id,valid,phases,type,provider,metrics,friendly_id,direction
c1f0a432-771e-46fc-aa84-efef5dd84c5e,awefice,m_1,"[2021-01-01 08:01:00+00:00, None)",A,CONSUMER,awe_voltage_reads,['V'],m_1,None
0d019e82-de4a-4115-9c64-e8288ade79e9,awefice,m_2,"[2021-01-01 08:01:00+00:00, None)",A,CONSUMER,awe_voltage_reads,['V'],m_2,None
adc65088-66bd-49ba-9deb-a42e538a50a6,awefice,m_3,"[2021-01-01 08:01:00+00:00, None)",A,CONSUMER,awe_voltage_reads,['V'],m_3,None
d351f2b5-3326-45a8-a259-9eaf439da74a,awefice,m_4,"[2021-01-01 08:01:00+00:00, None)",C,CONSUMER,awe_voltage_reads,['V'],m_4,None
b254dc80-07a9-464f-aedf-407450c07a82,awefice,m_5,"[2021-01-01 08:01:00+00:00, None)",C,CONSUMER,awe_voltage_reads,['V'],m_5,None


The `metrics` column is of type text[] and lists the units of measurements for time-series that can be retrieved for a given grid element. Please note that it is possible to have multiple metrics associated with the same `grid_element_data_source_id`. The following is an example of pulling out multiple grid element data sources available for the 'awefice' `grid_id` and the 'm_12' `grid_element_id`.

In [14]:
%%sql

SELECT geds.grid_element_data_source_id, 
        geds.grid_id, 
        geds.grid_element_id, 
        metric_key 
FROM grid_element_data_source geds
JOIN UNNEST(geds.metrics::TEXT[]) AS metric_key
    ON true
WHERE grid_id = 'awefice'
    AND grid_element_id = 'm_12';    

grid_element_data_source_id,grid_id,grid_element_id,metric_key
96ecaf9c-4c66-419d-a711-9d9027ff5f28,awefice,m_12,V
e3434ab9-c1aa-4d59-af40-e06efb196724,awefice,m_12,kWh


The `provider` column represents the origin of the stored data, and may be used to properly identify the grid_element_data_source for the time series of interest.

In [15]:
%%sql

SELECT distinct provider
FROM grid_element_data_source;

provider
awe_scada_reads
ohio_residential_meter
ohio_industrial_meter
csv_file
ohio_commercial_meter
awe_meter_reading
awe_ev_charger
awe_voltage_reads
third_party
awe_photovoltaic_v1


---

## Functions

**grid_get_downstream()**

The `grid_get_downstream()` function is one of tracing functions available in EDM, and is mentioned here to demonstrate a function with grid and grid element arguments. Its input arguments can be found by running the following query.

Please refer to the [grid_tracing.ipynb](../3_grid_tracing/grid_tracing.ipynb) notebook for examples of all tracing functions.


In [16]:
%%sql

SELECT function_args
FROM get_function_documentation('grid_get_downstream');

function_args
"grid_id text, grid_element_id text, include_self boolean DEFAULT true"


Below is an example with the 'awefice' `grid_id`, the 'm_10' `grid_element_id` and False to exclude the specified `grid_element_id` from the output list.

In [17]:
%%sql

SELECT *
FROM grid_get_downstream('awefice', 'm_10', False);

grid_id,grid_element_id,type,customer_type,phases,is_underground,is_producer,is_consumer,is_switchable,switch_is_open,terminal1_cn,terminal2_cn,power_flow_direction,upstream_grid_element_id,geometry,meta
awefice,batt_10,Battery,battery,ABC,None,True,True,False,None,con_135_t2,VCN.batt_10,both,None,0101000020E61000000B45A2539AC65EC0DDB431982CA24840,"{'name': 'batt_10', 'label': 'batt_10', 'sdp_id': None, 'lv_feeder': None, 'mv_feeder': None, 'ownership': 'private', 'make_model': 'LG CHEM RESU10H', 'enclosure_id': None, 'voltage_level': 'LV', 'capacity_usable': 9.6, 'commission_date': None, 'capacity_nominal': 9.8, 'manufacture_date': None, 'efficiency_round_trip': 97.96, 'max_peak_charge_power': 7.0, 'parent_transformer_id': 'transformer_36', 'max_peak_discharge_power': 7.0, 'max_continuous_charge_power': 5.0, 'max_continuous_discharge_power': 5.0}"
awefice,evch_10,EVCharger,ev_charger,ABC,None,True,True,False,None,con_135_t2,VCN.evch_10,both,None,0101000020E61000000B45A2539AC65EC0DDB431982CA24840,"{'name': None, 'label': 'evch_10', 'sdp_id': None, 'make_model': 'ChargePoint Home Flex [CPH50-NEMA14-50-L23]', 'max_current': 32, 'max_voltage': 240, 'is_dc_charger': False, 'voltage_level': 'LV', 'charging_level': 'level_2', 'connector_type': 'NEMA 14-30', 'active_charging_power': 7680, 'active_generation_power': 0}"
awefice,pv_10,Photovoltaic,photovoltaic,ABC,None,True,True,False,None,con_135_t2,VCN.pv_10,both,None,0101000020E61000000B45A2539AC65EC0DDB431982CA24840,"{'name': 'pv_10', 'label': 'pv_10', 'sdp_id': None, 'ac_size': 3600, 'dc_size': 2960, 'racking': 'roof', 'panel_area': 1.7272, 'tilt_angle': 42, 'azimuth_angle': 170, 'voltage_level': 'LV', 'dc_to_ac_ratio': 1.2162162162162162, 'commission_date': '2014-08-16', 'number_of_panels': 8, 'panel_make_model': 'LG Neon R [LG370Q1C-V5]', 'panel_rated_power': 370.0, 'degradation_annual': 0.3, 'generation_capacity': 3600, 'inverter_make_model': 'SolarEdge HDWave', 'degradation_25_years': 7.5}"


**ts_data_source_select()**

The `ts_data_source_select()` returns time series data for the given `id`, `metric_key`, and `timerange` input arguments.

Please note that the `id` argument refers to the `grid_element_data_source_id` from the `grid_element_data_source` table discussed in the above. And the `metric_key` is one of the metrics listed in the `metrics` column in the same table. Below is an example of retrieving V time series for the 'm_12' grid_element (whose grid_element_data_source_id is '96ecaf9c-4c66-419d-a711-9d9027ff5f28') for April 1 ~ 30, 2022 time range.

In [18]:
%%sql

SELECT timestamp, value
FROM ts_data_source_select('96ecaf9c-4c66-419d-a711-9d9027ff5f28', 'V', '[2022-04-01, 2022-04-30]'
                          )
LIMIT 5;

timestamp,value
2022-04-01 00:00:00+00:00,480.182
2022-04-01 01:00:00+00:00,480.068
2022-04-01 02:00:00+00:00,480.822
2022-04-01 03:00:00+00:00,480.423
2022-04-01 04:00:00+00:00,480.316


More generally, time series for multiple grid elements and metrics over the entire period can be retrieved by joining multiple tables as the following. Please note that the query output is ordered and limited for a demonstration purpose.

In [19]:
%%sql

SELECT geds.grid_element_id, 
        geds.type,
        metric_key, 
        tdss.timestamp, 
        tdss.value
FROM grid_element ge
JOIN grid_element_data_source geds
    ON geds.grid_id = ge.grid_id
    AND geds.grid_element_id = ge.grid_element_id
JOIN UNNEST(geds.metrics::TEXT[]) metric_key
    ON true
JOIN ts_data_source_select(geds.grid_element_data_source_id, metric_key) tdss
    ON true
WHERE ge.grid_id = 'awefice'
    AND ge.type = 'Meter'
    AND geds.type = 'CONSUMER' 
ORDER by 3
LIMIT 5;

grid_element_id,type,metric_key,timestamp,value
m_4,CONSUMER,kWh,2021-01-01 10:00:00+00:00,2.062
m_4,CONSUMER,kWh,2021-01-01 11:00:00+00:00,1.959
m_4,CONSUMER,kWh,2021-01-01 08:00:00+00:00,2.132
m_4,CONSUMER,kWh,2021-01-01 09:00:00+00:00,2.037
m_4,CONSUMER,kWh,2021-01-01 12:00:00+00:00,2.167


---

## SQL & Python

**Data Retrieval**

Below are a few ways to save SQL output into a local Python variable.

Option 1

In [20]:
%%sql result <<

SELECT grid_element_id, type
FROM grid_element
WHERE grid_id = 'awefice';

Returning data to local variable result


Option 2

In [21]:
# Save the sql output as a variable in one line.
result = %sql SELECT grid_element_id, type FROM grid_element WHERE grid_id = 'awefice'

Option 3

In [22]:
# Save the sql output as a variable in multiple lines.
result = %sql SELECT grid_element_id, type \
                FROM grid_element \
                WHERE grid_id = 'awefice'

Then the following code can be run to convert the above SQL output `result` into a Python dataframe to work with.

In [23]:
df = result.DataFrame()

# Return the first three lines of the dataframe to take a peek.
df.head(3)

,grid_element_id,type
0,batt_10,Battery
1,batt_19,Battery
2,batt_24,Battery


**Variable Input**

A python variable can be integrated into SQL queries by wrapping the variable with `'{}'`.

In [24]:
grid_element_id = 'm_14'

In [25]:
%%sql

SELECT *
FROM grid_element_data_source
WHERE grid_element_id = '{grid_element_id}';

grid_element_data_source_id,grid_id,grid_element_id,valid,phases,type,provider,metrics,friendly_id,direction
04f64b22-723b-446a-9640-9af7ca70e765,awefice,m_14,"[2021-01-01 08:01:00+00:00, None)",B,CONSUMER,awe_voltage_reads,['V'],m_14,None
16800cb1-4531-4f43-b5be-8b2d9caf78eb,awefice,m_14,"[2021-01-01 08:00:00+00:00, None)",B,CONSUMER,awe_meter_reading,['kWh'],m_14,None
